# Brazilian E-Commerce (Olist) – SQL Analysis Mini-Project

**Goal:** load, clean and analyze the Olist dataset using `pandas`, `SQLAlchemy` and SQL (SQLite).

**Contents**
1. Load and explore the data
2. Basic data cleaning
3. Create the SQLite database and export the DataFrames
4. SQL queries 1 to 9
5. Key insights

> Run the cells in order (Runtime → Run all).

## 1. Load and explore the data

In [ ]:
# Install kagglehub to download the public dataset directly in Colab
!pip install -q kagglehub

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, event, text
import kagglehub

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Download the dataset from Kaggle (no login needed for public datasets).
# Alternative: upload the CSV files manually and set DATA_DIR to their folder,
# e.g. DATA_DIR = "/content" or "/kaggle/input/brazilian-ecommerce".
DATA_DIR = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Files in dataset folder:", os.listdir(DATA_DIR))

In [ ]:
def read(name, **kwargs):
    """Read a CSV file from the dataset folder."""
    return pd.read_csv(os.path.join(DATA_DIR, name), **kwargs)

df_olist_customers = read("olist_customers_dataset.csv")
df_olist_sellers = read("olist_sellers_dataset.csv")
df_olist_order_reviews = read("olist_order_reviews_dataset.csv")
df_olist_order_items = read("olist_order_items_dataset.csv")
df_olist_products = read("olist_products_dataset.csv")
df_olist_geolocation = read("olist_geolocation_dataset.csv")
df_product_category_name_translation = read("product_category_name_translation.csv")
df_olist_orders = read("olist_orders_dataset.csv")
df_olist_order_payments = read("olist_order_payments_dataset.csv")

df_olist_customers.head()

In [ ]:
# Quick exploration: shape and missing values of every table
tables = {
    "customers": df_olist_customers,
    "sellers": df_olist_sellers,
    "order_reviews": df_olist_order_reviews,
    "order_items": df_olist_order_items,
    "products": df_olist_products,
    "geolocation": df_olist_geolocation,
    "category_translation": df_product_category_name_translation,
    "orders": df_olist_orders,
    "order_payments": df_olist_order_payments,
}

summary = pd.DataFrame({
    "rows": {k: v.shape[0] for k, v in tables.items()},
    "columns": {k: v.shape[1] for k, v in tables.items()},
    "missing_values": {k: int(v.isna().sum().sum()) for k, v in tables.items()},
    "duplicated_rows": {k: int(v.duplicated().sum()) for k, v in tables.items()},
})
summary

In [ ]:
# Missing values per column, only for columns that have any
for name, df in tables.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    if not nulls.empty:
        print(f"--- {name} ---")
        print(nulls.to_string(), "\n")

## 2. Basic data cleaning
- Convert date columns to `datetime` (so SQLite stores them as `YYYY-MM-DD HH:MM:SS` text and `strftime` works).
- Drop exact duplicate rows (the geolocation table has many).
- Fill missing product categories with `"unknown"`.

In [ ]:
# 2.1 Convert date columns to datetime
order_date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in order_date_cols:
    df_olist_orders[col] = pd.to_datetime(df_olist_orders[col])

for col in ["review_creation_date", "review_answer_timestamp"]:
    df_olist_order_reviews[col] = pd.to_datetime(df_olist_order_reviews[col])

df_olist_order_items["shipping_limit_date"] = pd.to_datetime(df_olist_order_items["shipping_limit_date"])

# 2.2 Drop exact duplicates
before = len(df_olist_geolocation)
df_olist_geolocation = df_olist_geolocation.drop_duplicates()
print(f"Geolocation rows: {before:,} -> {len(df_olist_geolocation):,}")

df_olist_customers = df_olist_customers.drop_duplicates()
df_olist_sellers = df_olist_sellers.drop_duplicates()
df_olist_order_items = df_olist_order_items.drop_duplicates()
df_olist_order_payments = df_olist_order_payments.drop_duplicates()

# 2.3 Fill missing product categories
df_olist_products["product_category_name"] = df_olist_products["product_category_name"].fillna("unknown")

# 2.4 Sanity check on order status values
df_olist_orders["order_status"].value_counts()

## 3. Create the SQLite database and export the DataFrames

In [ ]:
# In-memory SQLite database.
engine = create_engine("sqlite://", echo=False)

# Register a Haversine distance function (used in Query 9).
# SQLite has no built-in trigonometric functions in every build, so we add our own.
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in kilometers between two (lat, lon) points."""
    if None in (lat1, lon1, lat2, lon2):
        return None
    r = 6371.0088  # mean Earth radius in km
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = phi2 - phi1
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

@event.listens_for(engine, "connect")
def register_functions(dbapi_connection, connection_record):
    dbapi_connection.create_function("haversine_km", 4, haversine_km)

In [ ]:
# Export every DataFrame as a table in the SQLite database
df_olist_customers.to_sql("olist_customers", con=engine, index=False, if_exists="replace")
df_olist_sellers.to_sql("olist_sellers", con=engine, index=False, if_exists="replace")
df_olist_order_reviews.to_sql("olist_order_reviews", con=engine, index=False, if_exists="replace")
df_olist_order_items.to_sql("olist_order_items", con=engine, index=False, if_exists="replace")
df_olist_products.to_sql("olist_products_dataset", con=engine, index=False, if_exists="replace")
df_olist_geolocation.to_sql("olist_geolocation", con=engine, index=False, if_exists="replace", chunksize=50_000)
df_product_category_name_translation.to_sql("product_category_name_translation", con=engine, index=False, if_exists="replace")
df_olist_orders.to_sql("olist_orders", con=engine, index=False, if_exists="replace")
df_olist_order_payments.to_sql("olist_order_payments", con=engine, index=False, if_exists="replace")

print("Database created.")

In [ ]:
def run_query(sql, params=None):
    """Run a SQL query against the SQLite engine and return a DataFrame."""
    return pd.read_sql_query(text(sql), con=engine, params=params)

# Test: list the tables and select a few rows
print(run_query("SELECT name FROM sqlite_master WHERE type = 'table'")["name"].tolist())

sql = """
SELECT *
FROM olist_customers
LIMIT 5
"""
run_query(sql)

## 4. SQL queries

**Revenue definition used below:** the sum of `price` from `olist_order_items` (freight excluded), counting only orders with status `delivered`, unless stated otherwise.

### Query 1 – Count and percentage of orders purchased in Jan 2018 with a review score of 5
- Orders can have more than one review, so we count *distinct* orders.
- We report the percentage over (a) all orders purchased in Jan 2018 and (b) only those that were reviewed.

In [ ]:
# Query 1
query_1 = """
WITH jan_2018_orders AS (
    SELECT order_id
    FROM olist_orders
    WHERE strftime('%Y-%m', order_purchase_timestamp) = '2018-01'
),
order_reviews AS (
    SELECT j.order_id, r.review_score
    FROM jan_2018_orders j
    LEFT JOIN olist_order_reviews r ON r.order_id = j.order_id
)
SELECT
    COUNT(DISTINCT CASE WHEN review_score = 5 THEN order_id END)  AS orders_with_5_stars,
    COUNT(DISTINCT order_id)                                      AS total_orders_jan_2018,
    COUNT(DISTINCT CASE WHEN review_score IS NOT NULL THEN order_id END) AS reviewed_orders_jan_2018,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN review_score = 5 THEN order_id END)
          / COUNT(DISTINCT order_id), 2)                          AS pct_of_all_orders,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN review_score = 5 THEN order_id END)
          / COUNT(DISTINCT CASE WHEN review_score IS NOT NULL THEN order_id END), 2) AS pct_of_reviewed_orders
FROM order_reviews
"""
run_query(query_1)

### Query 2 – Customer purchase trend year-on-year
Orders and unique customers per year, plus the year-over-year growth (window function `LAG`).

In [ ]:
# Query 2
query_2 = """
WITH yearly AS (
    SELECT
        strftime('%Y', o.order_purchase_timestamp) AS year,
        COUNT(DISTINCT o.order_id)                 AS total_orders,
        COUNT(DISTINCT c.customer_unique_id)       AS unique_customers
    FROM olist_orders o
    JOIN olist_customers c ON c.customer_id = o.customer_id
    GROUP BY year
)
SELECT
    year,
    total_orders,
    unique_customers,
    ROUND(100.0 * (total_orders - LAG(total_orders) OVER (ORDER BY year))
          / LAG(total_orders) OVER (ORDER BY year), 2)         AS orders_yoy_growth_pct,
    ROUND(100.0 * (unique_customers - LAG(unique_customers) OVER (ORDER BY year))
          / LAG(unique_customers) OVER (ORDER BY year), 2)     AS customers_yoy_growth_pct
FROM yearly
ORDER BY year
"""
df_q2 = run_query(query_2)
display(df_q2)

# Monthly trend (extra view of the same question)
query_2b = """
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS year_month,
    COUNT(DISTINCT c.customer_unique_id)          AS unique_customers
FROM olist_orders o
JOIN olist_customers c ON c.customer_id = o.customer_id
GROUP BY year_month
ORDER BY year_month
"""
df_q2b = run_query(query_2b)
df_q2b.plot(x="year_month", y="unique_customers", figsize=(12, 4), marker="o",
            title="Unique customers per month", legend=False)
plt.xticks(rotation=90)
plt.ylabel("Unique customers")
plt.show()

### Query 3 – Average order value of customers
Order value = sum of `price + freight_value` of all items in the order.
First the overall average order value (AOV), then the AOV per customer (top 10).

In [ ]:
# Query 3 - overall average order value
query_3a = """
WITH order_totals AS (
    SELECT order_id, SUM(price + freight_value) AS order_value
    FROM olist_order_items
    GROUP BY order_id
)
SELECT
    COUNT(*)                     AS total_orders,
    ROUND(AVG(order_value), 2)   AS avg_order_value,
    ROUND(MIN(order_value), 2)   AS min_order_value,
    ROUND(MAX(order_value), 2)   AS max_order_value
FROM order_totals
"""
display(run_query(query_3a))

# Query 3 - average order value per customer (customers with 2+ orders, top 10)
query_3b = """
WITH order_totals AS (
    SELECT order_id, SUM(price + freight_value) AS order_value
    FROM olist_order_items
    GROUP BY order_id
)
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id)   AS number_of_orders,
    ROUND(AVG(t.order_value), 2) AS avg_order_value,
    ROUND(SUM(t.order_value), 2) AS total_spent
FROM olist_orders o
JOIN olist_customers c ON c.customer_id = o.customer_id
JOIN order_totals t    ON t.order_id = o.order_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY avg_order_value DESC
LIMIT 10
"""
run_query(query_3b)

### Query 4 – Top 5 cities with the highest revenue (2016–2018)

In [ ]:
# Query 4
query_4 = """
SELECT
    c.customer_city                 AS city,
    c.customer_state                AS state,
    COUNT(DISTINCT o.order_id)      AS total_orders,
    ROUND(SUM(i.price), 2)          AS revenue
FROM olist_orders o
JOIN olist_order_items i ON i.order_id = o.order_id
JOIN olist_customers c   ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
  AND strftime('%Y', o.order_purchase_timestamp) BETWEEN '2016' AND '2018'
GROUP BY c.customer_city, c.customer_state
ORDER BY revenue DESC
LIMIT 5
"""
df_q4 = run_query(query_4)
display(df_q4)

df_q4.plot(kind="bar", x="city", y="revenue", figsize=(8, 4), legend=False,
           title="Top 5 cities by revenue (2016-2018)")
plt.ylabel("Revenue (BRL)")
plt.show()

### Query 5 – State-wise revenue table (2016–2018)
One row per state, one column per year (pivot built with `CASE WHEN`).

In [ ]:
# Query 5
query_5 = """
SELECT
    c.customer_state AS state,
    ROUND(SUM(CASE WHEN strftime('%Y', o.order_purchase_timestamp) = '2016' THEN i.price ELSE 0 END), 2) AS revenue_2016,
    ROUND(SUM(CASE WHEN strftime('%Y', o.order_purchase_timestamp) = '2017' THEN i.price ELSE 0 END), 2) AS revenue_2017,
    ROUND(SUM(CASE WHEN strftime('%Y', o.order_purchase_timestamp) = '2018' THEN i.price ELSE 0 END), 2) AS revenue_2018,
    ROUND(SUM(i.price), 2) AS total_revenue
FROM olist_orders o
JOIN olist_order_items i ON i.order_id = o.order_id
JOIN olist_customers c   ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
  AND strftime('%Y', o.order_purchase_timestamp) BETWEEN '2016' AND '2018'
GROUP BY c.customer_state
ORDER BY total_revenue DESC
"""
run_query(query_5)

### Query 6 – Top successful sellers
Metrics per seller: goods sold, revenue, number of distinct customers and number of 5-star orders.
The base query is reused to rank sellers by different criteria.

In [ ]:
# Query 6 - base query (one row per seller with all metrics)
query_6_base = """
WITH seller_sales AS (
    SELECT
        i.seller_id,
        COUNT(*)                              AS goods_sold,
        ROUND(SUM(i.price), 2)                AS revenue,
        COUNT(DISTINCT c.customer_unique_id)  AS customer_count,
        COUNT(DISTINCT i.order_id)            AS total_orders
    FROM olist_order_items i
    JOIN olist_orders o    ON o.order_id = i.order_id
    JOIN olist_customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY i.seller_id
),
seller_orders AS (
    SELECT DISTINCT seller_id, order_id
    FROM olist_order_items
),
seller_five_star AS (
    SELECT so.seller_id, COUNT(DISTINCT so.order_id) AS five_star_orders
    FROM seller_orders so
    JOIN olist_order_reviews r ON r.order_id = so.order_id
    WHERE r.review_score = 5
    GROUP BY so.seller_id
)
SELECT
    s.seller_id,
    s.goods_sold,
    s.revenue,
    s.customer_count,
    s.total_orders,
    COALESCE(f.five_star_orders, 0) AS five_star_orders,
    ROUND(100.0 * COALESCE(f.five_star_orders, 0) / s.total_orders, 2) AS five_star_pct
FROM seller_sales s
LEFT JOIN seller_five_star f ON f.seller_id = s.seller_id
"""

# Top 10 sellers by revenue
query_6a = f"SELECT * FROM ({query_6_base}) ORDER BY revenue DESC LIMIT 10"
display(run_query(query_6a))

# Top 10 sellers by goods sold
query_6b = f"SELECT * FROM ({query_6_base}) ORDER BY goods_sold DESC LIMIT 10"
display(run_query(query_6b))

# Top 10 sellers by number of distinct customers
query_6c = f"SELECT * FROM ({query_6_base}) ORDER BY customer_count DESC LIMIT 10"
display(run_query(query_6c))

# Top 10 sellers with the most 5-star orders (at least 50 orders to avoid tiny samples)
query_6d = f"""
SELECT * FROM ({query_6_base})
WHERE total_orders >= 50
ORDER BY five_star_pct DESC, five_star_orders DESC
LIMIT 10
"""
run_query(query_6d)

### Query 7 – Delivery success rate across states
- **Delivery success rate:** delivered orders / all orders.
- **On-time rate:** delivered orders that arrived on or before the estimated date / delivered orders.

In [ ]:
# Query 7
query_7 = """
SELECT
    c.customer_state                                             AS state,
    COUNT(*)                                                     AS total_orders,
    SUM(CASE WHEN o.order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
    ROUND(100.0 * SUM(CASE WHEN o.order_status = 'delivered' THEN 1 ELSE 0 END) / COUNT(*), 2)
                                                                 AS delivery_success_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN o.order_status = 'delivered'
                            AND o.order_delivered_customer_date <= o.order_estimated_delivery_date
                           THEN 1 ELSE 0 END)
          / NULLIF(SUM(CASE WHEN o.order_status = 'delivered' THEN 1 ELSE 0 END), 0), 2)
                                                                 AS on_time_rate_pct
FROM olist_orders o
JOIN olist_customers c ON c.customer_id = o.customer_id
GROUP BY c.customer_state
ORDER BY delivery_success_rate_pct DESC, total_orders DESC
"""
df_q7 = run_query(query_7)
display(df_q7)

df_q7.sort_values("on_time_rate_pct").plot(kind="barh", x="state", y="on_time_rate_pct",
    figsize=(8, 8), legend=False, title="On-time delivery rate by state (%)")
plt.show()

### Query 8 – Preferred form of payment per product category
For every category we count distinct orders per payment type and keep the most used one (`ROW_NUMBER` window function).
Category names are translated to English.

In [ ]:
# Query 8 - preferred payment type per category
query_8 = """
WITH category_payments AS (
    SELECT
        COALESCE(t.product_category_name_english, p.product_category_name, 'unknown') AS category,
        pay.payment_type,
        COUNT(DISTINCT i.order_id) AS orders
    FROM olist_order_items i
    JOIN olist_products_dataset p ON p.product_id = i.product_id
    LEFT JOIN product_category_name_translation t
           ON t.product_category_name = p.product_category_name
    JOIN olist_order_payments pay ON pay.order_id = i.order_id
    GROUP BY category, pay.payment_type
),
ranked AS (
    SELECT
        category,
        payment_type,
        orders,
        SUM(orders) OVER (PARTITION BY category) AS category_total,
        ROW_NUMBER() OVER (PARTITION BY category ORDER BY orders DESC) AS rn
    FROM category_payments
)
SELECT
    category,
    payment_type AS preferred_payment_type,
    orders,
    ROUND(100.0 * orders / category_total, 2) AS share_pct
FROM ranked
WHERE rn = 1
ORDER BY orders DESC
"""
df_q8 = run_query(query_8)
display(df_q8.head(20))

# How many categories prefer each payment type?
df_q8["preferred_payment_type"].value_counts()

### Query 9 – Distance between cities
We average the latitude/longitude of every geolocation record per city and compute the Haversine distance
(`haversine_km`, registered in section 3) between each pair of cities in the list.
> City names in the geolocation table are not perfectly clean (accents, typos), so exact matches are used here.

In [ ]:
# Query 9 - pairwise distance (km) between a list of major cities
query_9 = """
WITH city_coords AS (
    SELECT
        geolocation_city       AS city,
        AVG(geolocation_lat)   AS lat,
        AVG(geolocation_lng)   AS lng
    FROM olist_geolocation
    WHERE geolocation_city IN ('sao paulo', 'rio de janeiro', 'belo horizonte', 'brasilia', 'curitiba')
    GROUP BY geolocation_city
)
SELECT
    a.city AS city_a,
    b.city AS city_b,
    ROUND(haversine_km(a.lat, a.lng, b.lat, b.lng), 1) AS distance_km
FROM city_coords a
JOIN city_coords b ON a.city < b.city
ORDER BY distance_km
"""
df_q9 = run_query(query_9)
display(df_q9)

In [ ]:
# Query 9 (extra) - average distance between the customer city and the seller city per order
# (uses the average coordinates of the zip code prefix)
query_9b = """
WITH zip_coords AS (
    SELECT geolocation_zip_code_prefix AS zip,
           AVG(geolocation_lat) AS lat,
           AVG(geolocation_lng) AS lng
    FROM olist_geolocation
    GROUP BY geolocation_zip_code_prefix
)
SELECT
    ROUND(AVG(haversine_km(cz.lat, cz.lng, sz.lat, sz.lng)), 1) AS avg_customer_seller_distance_km
FROM olist_orders o
JOIN olist_customers c  ON c.customer_id = o.customer_id
JOIN olist_order_items i ON i.order_id = o.order_id
JOIN olist_sellers s    ON s.seller_id = i.seller_id
JOIN zip_coords cz ON cz.zip = c.customer_zip_code_prefix
JOIN zip_coords sz ON sz.zip = s.seller_zip_code_prefix
"""
run_query(query_9b)

## 5. Key insights
Write your conclusions here after running the notebook, for example:
- **Growth:** how orders and customers changed between 2016, 2017 and 2018 (Query 2).
- **Geography:** which cities/states concentrate the revenue (Queries 4 and 5).
- **Customers:** average order value and repeat-customer behavior (Query 3).
- **Sellers:** what the top sellers have in common (Query 6).
- **Logistics:** states with the lowest on-time delivery rate (Query 7).
- **Payments:** the dominant payment method and the categories that differ (Query 8).
- **Distance:** relation between distance and delivery performance (Query 9).